# 02 · Versión A: preguntar al LLM de visión sin entrenarlo (*zero-shot*)

**¿Qué es un LLM de visión (VLM)?** Es un modelo de lenguaje al que se le ha «pegado» un ojo:

1. Un **codificador de imagen** (aquí SigLIP, 86 M parámetros) convierte la foto en vectores.
2. Un **conector** comprime esos vectores en 64 «palabras de imagen» del mismo tamaño que las palabras del texto.
3. Un **modelo de lenguaje** (aquí SmolLM2, 135 M parámetros) lee esas 64 palabras de imagen seguidas de la pregunta y escribe la respuesta, palabra a palabra.

Usamos **SmolVLM-256M-Instruct** (Hugging Face, licencia Apache-2.0): es de los VLM más pequeños que hay y cabe en un portátil sin GPU. *Zero-shot* significa que **no le enseñamos nada**: solo le describimos la tarea en el *prompt* y vemos qué contesta.

Este notebook lee `parejas/` de la carpeta de trabajo y deja `resultados/pred_A.jsonl` y `resultados/metricas_A.json`.

In [1]:
import os
N_EJEMPLOS = int(os.environ.get("ESTILO_N_EJEMPLOS", "60"))
EPOCAS = int(os.environ.get("ESTILO_EPOCAS", "2"))
RESOLUCION = int(os.environ.get("ESTILO_RESOLUCION", "224"))
MODELO = os.environ.get("ESTILO_MODELO", "HuggingFaceTB/SmolVLM-256M-Instruct")
CARPETA_TRABAJO = os.environ.get("ESTILO_TRABAJO", "_trabajo")

In [2]:
import json
import numpy as np
import matplotlib.pyplot as plt
import comun_estilo as ce

plt.rcParams["figure.dpi"] = 55
trabajo = ce.carpeta_trabajo(CARPETA_TRABAJO)   # fija también HF_HOME si no venía dado
hilos = ce.limitar_hilos()
import torch, transformers
print("torch", torch.__version__, "| transformers", transformers.__version__, "| hilos:", hilos)

with ce.Cronometro() as c:
    proc, vlm = ce.cargar_vlm(MODELO)
print(f"Modelo {MODELO} cargado en {c.s:.1f} s (incluye la descarga si no estaba en caché)")
for nombre, parte in vlm.model.named_children():
    print(f"  {nombre:14s} {sum(p.numel() for p in parte.parameters()) / 1e6:6.1f} M parámetros")

torch 2.15.0+cpu | transformers 5.23.0 | hilos: 12


Modelo HuggingFaceTB/SmolVLM-256M-Instruct cargado en 5.0 s (incluye la descarga si no estaba en caché)
  vision_model     86.4 M parámetros
  connector         7.1 M parámetros
  text_model      134.6 M parámetros


## 1. El *prompt*

Le explicamos qué ajustes queremos, en qué rango y que conteste **solo** con un JSON. Está en inglés porque el modelo se entrenó casi todo en inglés; las claves del JSON las dejamos en español porque son las de nuestro programa. `<image>` marca dónde entran las 64 palabras de imagen.

In [3]:
print(ce.texto_prompt(proc))

<|im_start|>User:<image>Look at the colors of this photo. Estimate the color edits that were applied to it and answer ONLY with a JSON object with these keys: "exposicion" (exposure in stops, -0.7 to 0.7), "temperatura" (white balance, -1 cool/blue to 1 warm/orange), "tinte" (tint, -1 green to 1 magenta), "contraste" (contrast factor, 0.6 to 1.5, 1 = unchanged), "saturacion" (saturation factor, 0.2 to 1.8, 1 = unchanged).<end_of_utterance>
Assistant:


## 2. Preguntar por cada foto de test

Si solo le damos el prompt, este modelo tan pequeño **ignora la orden y describe la foto** (lo vemos abajo con tres fotos). Un truco legítimo, que no le enseña nada, es **empezar nosotros la respuesta** con `{"exposicion":` para que siga escribiendo un JSON. Esa es la versión A oficial.

Para cada foto editada de test generamos la respuesta (sin azar: `do_sample=False`), intentamos **parsear** el JSON (aceptamos números entre comillas) y, si falta algún número, ponemos el valor neutro y lo contamos como fallo de formato.

In [4]:
test = ce.leer_parejas(trabajo, "test")
print("Sin empezar la respuesta (tres fotos):")
for f in test[:3]:
    print("  ->", ce.preguntar(proc, vlm, ce.abrir(trabajo, f["editada"]), max_tokens=40).replace("\n", " "))

INICIO = ' {"exposicion":'
resultados, tiempos = [], []
for f in test:
    with ce.Cronometro() as c:
        respuesta = ce.preguntar(proc, vlm, ce.abrir(trabajo, f["editada"]), inicio=INICIO)
    tiempos.append(c.s)
    pred, completo = ce.parsear(respuesta)
    resultados.append({"id": f["id"], "respuesta": respuesta, "pred": pred, "completo": completo})

(trabajo / "resultados").mkdir(exist_ok=True)
(trabajo / "resultados" / "pred_A.jsonl").write_text(
    "\n".join(json.dumps(r, ensure_ascii=False) for r in resultados), encoding="utf-8")
print(f"\nVersión A: {len(test)} fotos; {np.mean(tiempos):.2f} s por foto de media (mín {min(tiempos):.2f}, máx {max(tiempos):.2f})")

Sin empezar la respuesta (tres fotos):


  -> The image shows a dog running on a grassy field. The dog appears to be a mix of brown, white, and black, with a white and black coat. The dog's fur is short and


  -> The image shows a dog lying on a blanket, which appears to be a blanket with a pattern of white and gray. The dog is brown and appears to be relaxed, with its tongue hanging out and


  -> The dog's face is slightly turned away from the camera, and its ears are perked up. The dog's fur is mostly white with some black patches, and its eyes are dark brown. The



Versión A: 60 fotos; 5.05 s por foto de media (mín 2.67, máx 8.39)


## 3. ¿Cuánto se equivoca?

- **Formato correcto**: proporción de respuestas con los cinco números.
- **MAE (error medio absoluto)** de cada ajuste: media de |predicho − real|. Para poder comparar ajustes con rangos distintos damos también el **MAE normalizado** (MAE / ancho del rango): 0 es perfecto y, como referencia, contestar siempre el **valor neutro** da el valor que aparece en la última columna.

In [5]:
reales = [f["ajustes"] for f in test]
m = ce.error_por_ajuste([r["pred"] for r in resultados], reales)
neutro = ce.error_por_ajuste([ce.NEUTRO] * len(test), reales)
m["formato_correcto"] = float(np.mean([r["completo"] for r in resultados]))
m["segundos_por_foto"] = float(np.mean(tiempos))
m["memoria_pico_mb"] = ce.memoria_mb()
m["hilos"] = hilos
print(f"Formato correcto: {m['formato_correcto']:.0%}")
print(f"{'ajuste':12s} {'MAE':>6s} {'MAE norm.':>10s} {'neutro norm.':>13s}")
for k in ce.NOMBRES:
    print(f"{k:12s} {m['mae_'+k]:6.3f} {m['maen_'+k]:10.3f} {neutro['maen_'+k]:13.3f}")
print(f"{'MEDIA':12s} {'':6s} {m['mae_normalizado_medio']:10.3f} {neutro['mae_normalizado_medio']:13.3f}")
print("Memoria pico del proceso (MB):", m["memoria_pico_mb"])
# Se guarda en JSON; el run de MLflow de A lo crea el notebook 04 (así, en CI, una sola base de MLflow)
(trabajo / "resultados" / "metricas_A.json").write_text(json.dumps(m, indent=1), encoding="utf-8")

Formato correcto: 55%
ajuste          MAE  MAE norm.  neutro norm.
exposicion    0.683      0.488         0.262
temperatura   0.984      0.492         0.250
tinte         0.865      0.432         0.249
contraste     0.335      0.372         0.242
saturacion    0.608      0.380         0.290
MEDIA                    0.433         0.259
Memoria pico del proceso (MB): 1541.2


547

## 4. Respuestas reales: buenas y malas

Mostramos tal cual lo que escribió el modelo (con nuestro arranque `{"exposicion":` incluido). «Buenas» son las que tienen el formato correcto, aunque los números puedan estar mal; «malas», las que no se pudieron leer enteras. Fíjate si los números dependen de la foto o si el modelo **copia los extremos de los rangos que le dimos en el prompt**.

In [6]:
buenas = [r for r in resultados if r["completo"]][:3]
malas = [r for r in resultados if not r["completo"]][:3]
por_id = {f["id"]: f for f in test}
for titulo, grupo in (("BUENAS (formato correcto)", buenas), ("MALAS (no se pudo leer el JSON)", malas)):
    print("=" * 20, titulo, "=" * 20)
    for r in grupo:
        print("real     :", ce.a_json(por_id[r["id"]]["ajustes"]))
        print("respuesta:", r["respuesta"][:300].replace("\n", " "))
        print()
    if not grupo:
        print("(ninguna)\n")

==================== BUENAS (formato correcto) ====================
real     : {"exposicion": 0.45, "temperatura": 0.5, "tinte": 0.03, "contraste": 1.35, "saturacion": 1.69}
respuesta: {"exposicion": "0.7", "temperatura": "-1", "tinte": "-1", "contraste": "-1", "saturacion": "0.2", "tinte_saturation": "1.8", "tinte_contraste": "1.5", "saturacion_contraste": "1.5", "satur

real     : {"exposicion": 0.0, "temperatura": -0.39, "tinte": 0.35, "contraste": 1.23, "saturacion": 0.43}
respuesta: {"exposicion": "0.7", "temperatura": "-1", "tinte": "-1", "tinte_contraste": "-1", "saturacion": "0.2", "tinte_contraste_saturation": "1", "saturacion_saturation_color": "0.2", "tinte_color_saturation

real     : {"exposicion": 0.37, "temperatura": 0.84, "tinte": 0.99, "contraste": 0.67, "saturacion": 1.58}
respuesta: {"exposicion": "0.7", "temperatura": "-1", "tinte": "-1", "tinte_contraste": "0.6", "saturacion": "0.2", "tinte_contraste_saturation": "1", "saturation_factor": "0.2", "tinte_contraste_sa

**Conclusión de A.** El resultado de arriba es el punto de partida: lo que el modelo sabe hacer sin ver ni un solo ejemplo. En el notebook 03 le enseñaremos la tarea con LoRA usando las parejas fabricadas.